# Vectorization And Features

This notebook is the eighth step in the operator sequence. The earlier notebooks built and manipulated `AbstractGraph` structures. Here we turn those structures into numerical feature matrices that can be passed into downstream ML workflows.

Previous: [07 XML And Operator Serialization](./example_abstract_graph_operators_07_xml_and_operator_serialization.ipynb)  
Next: [09 Preprocessor Attention Pipeline](./example_abstract_graph_operators_09_preprocessor_attention_pipeline.ipynb)


## What vectorization means here

After a decomposition pipeline builds an interpretation graph, `abstractgraph.vectorize` converts the result into a fixed-width matrix. Rows correspond to base nodes, and columns correspond to hashed feature buckets.

Two details matter immediately:

1. column `0` is a bias feature filled with ones
2. column `1` stores the base node degree

The remaining columns hold hashed subgraph-label counts.


In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import networkx as nx
from IPython.core.display import HTML
from warnings import simplefilter

simplefilter(action='ignore', category=FutureWarning)
HTML('<style>.container { width:95% !important; }</style>')


In [ ]:
from abstractgraph.graphs import graph_to_abstract_graph
from abstractgraph.display import display, display_decomposition_graph, display_graph, display_mappings
from abstractgraph.operators import *
from abstractgraph.vectorize import vectorize, AbstractGraphTransformer, AbstractGraphNodeTransformer


def draw(graph, decomposition_function, *, nbits=6, size=(12, 6), n_elements_per_row=8):
    display_decomposition_graph(decomposition_function)
    ag = graph_to_abstract_graph(graph, decomposition_function=decomposition_function, nbits=nbits)
    display(ag, size=size)
    display_mappings(ag, n_elements_per_row=n_elements_per_row)
    return ag


def nonzero_columns(row):
    idx = np.flatnonzero(row)
    return pd.DataFrame({'column': idx, 'value': row[idx]})


## A small family of toy graphs

We use one base graph plus two small variants so we can compare graph-level feature vectors across a batch.


In [ ]:
def make_graph(branch=False, extra_triangle=False):
    graph = nx.Graph()
    graph.add_nodes_from([
        (0, {'label': 'C'}),
        (1, {'label': 'C'}),
        (2, {'label': 'O'}),
        (3, {'label': 'N'}),
        (4, {'label': 'C'}),
        (5, {'label': 'S'}),
        (6, {'label': 'C'}),
        (7, {'label': 'O'}),
    ])
    graph.add_edges_from([
        (0, 1, {'label': 'ring'}),
        (1, 2, {'label': 'ring'}),
        (2, 3, {'label': 'ring'}),
        (3, 0, {'label': 'ring'}),
        (2, 4, {'label': 'bridge'}),
        (4, 5, {'label': 'ring'}),
        (5, 2, {'label': 'ring'}),
        (3, 6, {'label': 'branch'}),
        (6, 7, {'label': 'branch'}),
    ])
    if branch:
        graph.add_node(8, label='Cl')
        graph.add_edge(5, 8, label='branch')
    if extra_triangle:
        graph.add_node(8, label='C')
        graph.add_node(9, label='N')
        graph.add_edges_from([
            (1, 8, {'label': 'ring'}),
            (8, 9, {'label': 'ring'}),
            (9, 1, {'label': 'ring'}),
        ])
    return graph

base_graph = make_graph()
branch_graph = make_graph(branch=True)
triangle_graph = make_graph(extra_triangle=True)

display_graph(base_graph)


## Build an `AbstractGraph` first

Vectorization is downstream of decomposition, so we start with a pipeline that mixes several unary operators.


In [ ]:
df = compose(intersection_edges(), add(node(), edge(), cycle()))
ag = draw(base_graph, df, nbits=6)


## Direct node-level vectorization

Calling `vectorize(...)` on an `AbstractGraph` returns one feature row per base node.


In [ ]:
X_dense = vectorize(ag, nbits=6, return_dense=True)
print(X_dense.shape)
pd.DataFrame(X_dense).head()


## Inspect one row

The first two nonzero columns should always include the bias term and the node degree. The remaining active columns depend on which abstract substructures involve that node.


In [ ]:
nonzero_columns(X_dense[0])


## Sparse output

The same vectorization can be returned as a CSR matrix when you want a more storage-efficient representation.


In [ ]:
X_sparse = vectorize(ag, nbits=6, return_dense=False)
print(type(X_sparse).__name__)
print(X_sparse.shape)
print('nnz =', X_sparse.nnz)
X_sparse


## Different decompositions give different feature matrices

Changing the decomposition changes the vocabulary, which changes the final feature counts.


In [ ]:
for name, candidate_df in {
    'node_only': compose(node()),
    'neighborhood_radius_1': compose(neighborhood(radius=1)),
    'node_edge_cycle': compose(intersection_edges(), add(node(), edge(), cycle())),
}.items():
    candidate_ag = graph_to_abstract_graph(base_graph, decomposition_function=candidate_df, nbits=6)
    candidate_X = vectorize(candidate_ag, nbits=6, return_dense=True)
    print(name, candidate_X.shape, 'sum =', float(candidate_X.sum()))


## `AbstractGraphNodeTransformer`

The node-level transformer packages the same logic for batches of graphs. It returns one feature matrix per input graph.


In [ ]:
node_transformer = AbstractGraphNodeTransformer(
    nbits=6,
    decomposition_function=df,
    return_dense=True,
    n_jobs=1,
)
node_feature_matrices = node_transformer.transform([base_graph, branch_graph])
[arr.shape for arr in node_feature_matrices]


## `AbstractGraphTransformer`

The graph-level transformer sums node-level features into one feature vector per graph. This is often the more convenient representation for classical ML estimators.


In [ ]:
graph_transformer = AbstractGraphTransformer(
    nbits=6,
    decomposition_function=df,
    return_dense=True,
    n_jobs=1,
    backend='threading',
)
X_graph = graph_transformer.transform([base_graph, branch_graph, triangle_graph])
print(X_graph.shape)
pd.DataFrame(X_graph, index=['base', 'branch', 'triangle']).iloc[:, :12]


## Compare graph-level totals

The variants are small, but their feature totals differ because their abstract vocabularies differ.


In [ ]:
pd.Series(X_graph.sum(axis=1), index=['base', 'branch', 'triangle'], name='feature_sum')


## Practical reading

A useful mental model is:

- operators define the structural vocabulary
- hashing compresses that vocabulary into a bounded feature space
- `vectorize(...)` maps the current `AbstractGraph` into numeric form
- transformers package the same logic for batches of graphs


## Summary

This notebook completed the path from graph decomposition to fixed-width features. At this point the staged sequence covers the core operator language plus the main route into ML-ready matrices.

A natural next notebook is `08_preprocessor_attention_pipeline.ipynb`, which would shift from operator design to base-graph construction with the repository’s preprocessing utilities.

Previous: [07 XML And Operator Serialization](./example_abstract_graph_operators_07_xml_and_operator_serialization.ipynb)  
Next: [09 Preprocessor Attention Pipeline](./example_abstract_graph_operators_09_preprocessor_attention_pipeline.ipynb)
